In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix
)

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
OUTPUT_DIR = Path("../reports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("08 — SKEWNESS ANALYSIS: Anomaly Feature Cleaning")
print("    min_seg_size_forward + Bwd Header Length Removal")
print("=" * 60)

In [ ]:
FEATURED_PATH  = "../data/csv/featured_dataset.csv"
FEATURES_TXT   = "../data/csv/selected_features.txt"
LABEL_MAP_PATH = "../data/csv/label_mapping.csv"

with open(FEATURES_TXT) as f:
    FEATURE_COLS = [line.strip() for line in f if line.strip()]

label_mapping = pd.read_csv(LABEL_MAP_PATH)
INT_TO_LABEL  = dict(zip(label_mapping["label_int"], label_mapping["label_string"]))

df = pd.read_csv(FEATURED_PATH, low_memory=False)
print(f"Dataset: {df.shape[0]:,} rows, {len(FEATURE_COLS)} features")

In [ ]:
# Define both anomaly features in a single list
ANOMALY_FEATURES = ["min_seg_size_forward", "Bwd Header Length"]

print("=" * 65)
print("ANOMALY FEATURE CHECK REPORT")
print("  (Physically impossible values caused by Wireshark/CICFlowMeter)")
print("=" * 65)

for feat in ANOMALY_FEATURES:
    in_list = feat in FEATURE_COLS
    if in_list:
        skew_val = df[feat].skew()
        print(f"  ✓ '{feat}' was FOUND in the feature list")
        print(f"      Skewness value : {skew_val:.2f}")
        print(f"      Expected range : |skew| <= 10 (normal)")
        print(f"      Detected       : Physically impossible extreme negative skew")
        print(f"      Decision       : WILL BE DROPPED")
    else:
        print(f"  [WARNING] '{feat}' was not found in the feature list.")
        print(f"      It may have been removed by the 03_feature_engineering correlation filter.")
    print()

# Actual list to drop (only existing features)
features_to_drop = [f for f in ANOMALY_FEATURES if f in FEATURE_COLS]
print(f"Total anomaly features to drop: {len(features_to_drop)}")
for f in features_to_drop:
    print(f"  - {f}")

In [ ]:
def train_evaluate_rf(X, y, experiment_name, int_to_label=None):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )
    model = RandomForestClassifier(
        n_estimators=100, random_state=RANDOM_STATE, n_jobs=2
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics = {
        "experiment":   experiment_name,
        "n_features":   X.shape[1],
        "accuracy":     accuracy_score(y_test, y_pred),
        "precision_w":  precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall_w":     recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_weighted":  f1_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_macro":     f1_score(y_test, y_pred, average="macro", zero_division=0),
    }
    return model, metrics, y_test, y_pred


def plot_confusion_matrix(y_true, y_pred, title, save_path, int_to_label=None):
    labels = sorted(set(y_true) | set(y_pred))
    if int_to_label:
        display_labels = [int_to_label.get(l, str(l)) for l in labels]
    else:
        display_labels = [str(l) for l in labels]

    cm = confusion_matrix(y_true, y_pred, labels=labels)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    for ax, data, fmt, subtitle in [
        (axes[0], cm,      ",d",  "Raw Counts"),
        (axes[1], cm_norm, ".2f", "Normalized (Recall-based)")
    ]:
        sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                    xticklabels=display_labels, yticklabels=display_labels,
                    ax=ax, annot_kws={"size": 7})
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
        ax.set_title(subtitle)
        ax.tick_params(axis='x', rotation=45, labelsize=7)
        ax.tick_params(axis='y', rotation=0,  labelsize=7)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved: {save_path}")

In [ ]:
import re

# First remove leakage features (same as in 07)
LEAKAGE_PATTERNS = [
    r"source.?address", r"destination.?address",
    r"src.?ip", r"dst.?ip",
    r"source.?ip", r"destination.?ip",
    r"flow.?id", r"timestamp",
]
PORT_PATTERNS = [r"^destination.?port$"]

def find_matching_columns(columns, patterns):
    found = []
    for col in columns:
        col_norm = col.lower().strip().replace(" ", "_")
        for pat in patterns:
            if re.search(pat, col_norm):
                found.append(col)
                break
    return sorted(set(found))

leakage_cols = find_matching_columns(FEATURE_COLS, LEAKAGE_PATTERNS)
port_cols    = find_matching_columns(FEATURE_COLS, PORT_PATTERNS)
all_leakage  = sorted(set(leakage_cols + port_cols))

# Feature list with leakage removed
# (min_seg_size_forward and Bwd Header Length are still included at this stage)
features_no_leak = [f for f in FEATURE_COLS if f not in all_leakage]

print("=" * 65)
print("EXPERIMENT 1: Anomaly features INCLUDED (leakage removed)")
print("  → min_seg_size_forward  ≈ -681 skewness  (Wireshark bug)")
print("  → Bwd Header Length     ≈ -578 skewness  (CICFlowMeter anomaly)")
print("=" * 65)
print(f"  Number of features: {len(features_no_leak)}")

y_mc = df["label_multiclass"].values
X_before = df[features_no_leak].values

_, metrics_before, yt_b, yp_b = train_evaluate_rf(
    X_before, y_mc, "BEFORE_anomaly_removal", INT_TO_LABEL
)

print(f"  Accuracy       : {metrics_before['accuracy']:.4f}")
print(f"  F1 Weighted    : {metrics_before['f1_weighted']:.4f}")
print(f"  F1 Macro       : {metrics_before['f1_macro']:.4f}")

plot_confusion_matrix(
    yt_b, yp_b,
    "Confusion Matrix — BEFORE Anomaly Feature Removal\n"
    "(min_seg_size_forward + Bwd Header Length included)",
    OUTPUT_DIR / "confusion_skew_before.png",
    INT_TO_LABEL
)

In [ ]:
print("=" * 65)
print("EXPERIMENT 2: Anomaly features REMOVED")
print("  → min_seg_size_forward  : ≈ -681 skewness — Wireshark bug")
print("  → Bwd Header Length     : ≈ -578 skewness — CICFlowMeter anomaly")
print("=" * 65)

# Remove both anomaly features from the list
features_to_drop = ["min_seg_size_forward", "Bwd Header Length"]

features_no_skew = [f for f in features_no_leak if f not in features_to_drop]

dropped_actual = [f for f in features_to_drop if f in features_no_leak]
not_found      = [f for f in features_to_drop if f not in features_no_leak]

for f in dropped_actual:
    print(f"  ✓ '{f}' removed.")
for f in not_found:
    print(f"  [INFO] '{f}' was already not in the list, skipping.")

print(f"\n  Remaining feature count: {len(features_no_skew)}")

X_after = df[features_no_skew].values

_, metrics_after, yt_a, yp_a = train_evaluate_rf(
    X_after, y_mc, "AFTER_anomaly_removal", INT_TO_LABEL
)

print(f"  Accuracy       : {metrics_after['accuracy']:.4f}")
print(f"  F1 Weighted    : {metrics_after['f1_weighted']:.4f}")
print(f"  F1 Macro       : {metrics_after['f1_macro']:.4f}")

plot_confusion_matrix(
    yt_a, yp_a,
    "Confusion Matrix — AFTER Anomaly Feature Removal\n"
    "(min_seg_size_forward + Bwd Header Length removed)",
    OUTPUT_DIR / "confusion_skew_after.png",
    INT_TO_LABEL
)

In [ ]:
print("=" * 65)
print("SKEWNESS / ANOMALY COMPARISON TABLE")
print("  BEFORE: min_seg_size_forward + Bwd Header Length included")
print("  AFTER : both anomaly features removed")
print("=" * 65)

comp_skew = pd.DataFrame([metrics_before, metrics_after])
comp_skew["delta_accuracy"]    = comp_skew["accuracy"]    - metrics_before["accuracy"]
comp_skew["delta_f1_weighted"] = comp_skew["f1_weighted"] - metrics_before["f1_weighted"]
comp_skew["delta_f1_macro"]    = comp_skew["f1_macro"]    - metrics_before["f1_macro"]

display_cols = ["experiment", "n_features", "accuracy", "precision_w", "recall_w",
                "f1_weighted", "f1_macro", "delta_accuracy", "delta_f1_weighted", "delta_f1_macro"]

print(comp_skew[display_cols].round(4).to_string(index=False))

comp_skew.to_csv(OUTPUT_DIR / "skewness_comparison.csv", index=False)
print(f"\n✓ Saved: {OUTPUT_DIR / 'skewness_comparison.csv'}")